# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [21]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", "YES" if hf_token else "NO")

Token loaded: YES


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [22]:
from huggingface_hub import hf_hub_download
import duckdb

con = duckdb.connect()

dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token
)

dim_clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=hf_token
)

fact_march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

In [17]:
grain_check = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) as unique_combos
    FROM '{fact_march_path}'
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378


In [23]:
date_span = con.execute(f"""
    SELECT
        COUNT(*) as row_count,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM '{fact_march_path}'
""").df()
print(date_span)

   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31


In [24]:
availability = con.execute(f"""
    SELECT
        COUNT(*) as rows_before,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4
    FROM '{fact_march_path}'
""").df()
print(availability)

   rows_before  rows_with_gsc  rows_with_ga4
0      9841378      3611061.0       413966.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
features = con.execute(f"""
    SELECT
        dc.content_hash_id,
        dc.client_hash_id,
        dc.search_volume,
        dc.competition,
        dc.backlinks,
        dc.word_count,
        AVG(f.gsc_avg_position) as avg_position_march
    FROM '{dim_content_path}' dc
    JOIN '{fact_march_path}' f
        ON dc.content_hash_id = f.content_hash_id
        AND dc.client_hash_id = f.client_hash_id
    WHERE dc.is_deleted IS FALSE
    GROUP BY dc.content_hash_id, dc.client_hash_id, dc.search_volume,
             dc.competition, dc.backlinks, dc.word_count
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,search_volume,competition,backlinks,word_count,avg_position_march
0,content_05597932fe4da067,client_73cda7b4e4f265ea,10,1.00,<NA>,<NA>,2.714744
1,content_05434271b257bb68,client_73cda7b4e4f265ea,10,0.00,<NA>,<NA>,6.320337
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,50,0.06,<NA>,2475,4.459107
3,content_22610b0934f8825e,client_73cda7b4e4f265ea,110,0.37,<NA>,<NA>,12.791667
4,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,40,0.89,<NA>,<NA>,9.445635


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
from scipy.stats import spearmanr
import numpy as np

features['honest_score'] = (
    features['search_volume'].rank()
    - features['competition'].rank()
    + features['avg_position_march'].rank()
)
features['honest_rank'] = features['honest_score'].rank(ascending=False)
features[['content_hash_id','honest_score','honest_rank']].head()

,content_hash_id,honest_score,honest_rank
0,content_05597932fe4da067,-39266.0,160551.0
1,content_05434271b257bb68,191057.0,38794.0
2,content_d056587ff7faca0c,132057.5,83300.0
3,content_22610b0934f8825e,180569.0,43834.0
4,content_5d412fba6e1a2582,119908.0,92521.0


In [27]:
fact_april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=hf_token
)

future_leak = con.execute(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) as future_clicks
    FROM '{fact_april_path}'
    GROUP BY content_hash_id, client_hash_id
""").df()

features_leaked = features.merge(future_leak, on=['content_hash_id','client_hash_id'], how='left')
features_leaked['leaky_score'] = features_leaked['future_clicks']
features_leaked['leaky_rank'] = features_leaked['leaky_score'].rank(ascending=False)

corr = features_leaked[['honest_score','future_clicks']].corr().iloc[0,1]
print(corr)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-0.04796692645805546


In [28]:
features_final = features_leaked.drop(columns=['future_clicks','leaky_score','leaky_rank'])
features_final.head()

,content_hash_id,client_hash_id,search_volume,competition,backlinks,word_count,avg_position_march,honest_score,honest_rank
0,content_05597932fe4da067,client_73cda7b4e4f265ea,10,1.00,<NA>,<NA>,2.714744,-39266.0,160551.0
1,content_05434271b257bb68,client_73cda7b4e4f265ea,10,0.00,<NA>,<NA>,6.320337,191057.0,38794.0
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,50,0.06,<NA>,2475,4.459107,132057.5,83300.0
3,content_22610b0934f8825e,client_73cda7b4e4f265ea,110,0.37,<NA>,<NA>,12.791667,180569.0,43834.0
4,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,40,0.89,<NA>,<NA>,9.445635,119908.0,92521.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.